In [15]:
import os
import re
import glob
import pandas as pd
from pathlib import Path
from slap2_processing.src import generate_instrument_json as gij
from slap2_processing.src.generate_acquisition_json import Slap2VCOAcquisitionEtl

In [16]:
basepath = r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
summary_path = glob.glob(os.path.join(basepath,'**summary.xlsx'))[0]
summary_df = pd.read_excel(summary_path,sheet_name='subjects')
session_df = pd.read_excel(summary_path,sheet_name='sessions')

In [17]:
session_df

,session_id,subject_id,session_#,session_date,indicator1,indicator2,dmd1_depth,dmd2_depth,paradigm,session_type,...,instrument_id,camera_type,has raster ROI?,has integration roi?,behavior_rig,quality,flags,session_dir,purpose,notes
0,803496_2025-07-01_14-56-10,803496,1,2025-07-01,iGluSnFR4,NaN,25,100,change_detection_passive,expression_check,...,SLAP2_1,NaN,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
1,809436_2025-07-25_13-02-10,803496,2,2025-07-25,iGluSnFR4,NaN,25,100,change_detection_passive,familiar,...,SLAP2_1,vimba,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
2,809436_2025-07-28_08-04-39,803496,3,2025-07-28,iGluSnFR4,NaN,25,100,change_detection_passive,familiar,...,SLAP2_1,vimba,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
3,803496_2025-07-29_13-34-35,803496,4,2025-07-29,iGluSnFR4,NaN,25,100,change_detection_passive,familiar,...,SLAP2_1,vimba,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
4,803496_2025-07-30_10-05-23,803496,5,2025-07-30,iGluSnFR4,NaN,25,100,change_detection_passive,novel,...,SLAP2_1,vimba,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,838410_2026-03-08_14-02-23,838410,7,2026-03-08,iGluSnFR4,RCaMP3,50,200,change_detection_passive,novel+,...,SLAP2_1,spinnaker,yes,no,VCO.1,good,Acquisition saved to a single file,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
102,838410_2026-03-09_11-04-01,838410,8,2026-03-09,iGluSnFR4,RCaMP3,50,200,change_detection_passive,novel+,...,SLAP2_1,spinnaker,yes,no,VCO.1,poor,Too much motion on DMD2,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
103,836173_2026-02-13_16-42-29,836173,1,2026-02-13,iGluSnFR4,RCaMP3,50,200,change_detection_passive,expression_check,...,SLAP2_1,NaN,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
104,836174_2026-02-13_16-17-06,836174,1,2026-02-13,iGluSnFR4,RCaMP3,50,200,change_detection_passive,expression_check,...,SLAP2_1,NaN,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN


In [18]:
def find_session_root_subdir(session_dir: Path, mouse: int | str) -> Path:
    """
    Find the subdirectory inside `session_dir` whose name matches:
        nnnnnn_yyyy-mm-dd_hh-mm-ss
    where nnnnnn == mouse id (6 digits).
    Returns the matching Path, or raises ValueError with a helpful message.
    """
    mouse = str(mouse)
    pattern = re.compile(rf"^{re.escape(mouse)}_\d{{4}}-\d{{2}}-\d{{2}}_\d{{2}}-\d{{2}}-\d{{2}}$")

    matches = [p for p in session_dir.iterdir() if p.is_dir() and pattern.match(p.name)]
    if len(matches) == 0:
        raise ValueError(f"No matching destination subdir in {session_dir}")
    if len(matches) > 1:
        raise ValueError(f"Multiple matching destination subdirs in {session_dir}: {[m.name for m in matches]}")
    return matches[0]

### Generate instrument.json

In [38]:
for idx, row in session_df.iterrows():
    try:
        sess = find_session_root_subdir(Path(row['session_dir']), mouse=row['subject_id'])
    except Exception:
        sess = Path(row['session_dir'])

    out_path = Path(sess) / "instrument.json"
    print(f"Attempting to generate instrument.json for {Path(sess).stem}")

    try:
        gij.generate_instrument_json(
            Path(sess),
            use_vimba=(row['camera_type'] == 'vimba')
        )

        if out_path.exists():
            print(f"SUCCESS: wrote {out_path}")
        else:
            print(f"NO FILE FOUND AFTER RUN: expected {out_path}")

    except BaseException as e:
        print(f"FAILED: {type(e).__name__}: {e}")

    print()

Attempting to generate instrument.json for 803496_2025-07-01_14-56-10
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-01_803496\803496_2025-07-01_14-56-10
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-01_803496\803496_2025-07-01_14-56-10\instrument.json

Attempting to generate instrument.json for 803496_2025-07-25_13-02-10
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\803496_2025-07-25_13-02-10
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-25_803496\803496_2025-07-25_13-02-10\instrument.json

Attempting to generate instrument.json for 803496_2025-07-28_08-04-39
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803496\2025-07-28_803496\803496_2025-07-28_08-04-39
SU

Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\810196\2025-07-28_810196\810196_2025-07-28_19-59-05
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\810196\2025-07-28_810196\810196_2025-07-28_19-59-05\instrument.json

Attempting to generate instrument.json for 810196_2025-07-29_17-02-41
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\810196\2025-07-29_810196\810196_2025-07-29_17-02-41
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\810196\2025-07-29_810196\810196_2025-07-29_17-02-41\instrument.json

Attempting to generate instrument.json for 810196_2025-07-31_08-28-08
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\810196\2025-07-30_810196\810196_2025-07-31_08-28-08
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\i

Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\structure_volume\803121\803121_2026-02-19_11-06-55
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\structure_volume\803121\803121_2026-02-19_11-06-55\instrument.json

Attempting to generate instrument.json for 814591_2025-11-07_10-21-49
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\structure_volume\814591\2025-11-07_814591\814591_2025-11-07_10-21-49
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\structure_volume\814591\2025-11-07_814591\814591_2025-11-07_10-21-49\instrument.json

Attempting to generate instrument.json for 814591_2026-01-13_10-37-24
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\structure_volume\814591\2026-01-13_814591_BigStacks\814591_2026-01-13_10-37-24
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynami

Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826032\826032_2026-02-03_16-02-38
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826032\826032_2026-02-03_16-02-38\instrument.json

Attempting to generate instrument.json for 826032_2026-02-04_13-49-07
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826032\826032_2026-02-04_13-49-07
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826032\826032_2026-02-04_13-49-07\instrument.json

Attempting to generate instrument.json for 826032_2026-02-05_13-18-03
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826032\826032_2026-02-05_13-18-03
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826032\826032_2026-02-05_13-18-03\instrument.json

Attempting to generate instrument.json for 826032_2026-02-0

Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\834788\834788_2026-03-09_09-07-35
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\834788\834788_2026-03-09_09-07-35\instrument.json

Attempting to generate instrument.json for 838410_2026-02-13_15-49-00
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-02-13_15-49-00
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-02-13_15-49-00\instrument.json

Attempting to generate instrument.json for 838410_2026-03-02_12-40-55
Successfully wrote instrument JSON file to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-02_12-40-55
SUCCESS: wrote \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\838410\838410_2026-03-02_12-40-55\inst

### Get stimulus parameters

In [20]:
stim_path = r"C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\Aind.Behavior.ChangeDetection\src\stimuli\images_B"

In [21]:
image_names = os.listdir(stim_path)
image_names

['100075.tiff',
 '216066.tiff',
 '268048.tiff',
 '41006.tiff',
 '69022.tiff',
 'imk01306.tiff',
 'imk01333.tiff',
 'McGill_stairs.tiff']

### Generate acquisition.json

In [37]:
mouse = 803121
process_dirs = session_df[(session_df['subject_id']==mouse)
                          &(session_df['session_type']!='expression_check')
                          &(session_df['session_type']!='volume_imaging')]['session_dir'].values
session_dirs = [find_session_root_subdir(Path(sess),mouse) for sess in process_dirs]
results = []

for sess_dir in session_dirs:
    try:
        etl = Slap2VCOAcquisitionEtl(
            asset_dir=sess_dir,
            output_dir=sess_dir,      
            timing_source="harp",     
            timing_source_column=None,
            save_images=False,
        )
        etl.run_job()

        results.append({
            "session_dir": str(sess_dir),
            "status": "success",
            "error": ""
        })

    except Exception as e:
        results.append({
            "session_dir": str(sess_dir),
            "status": "failed",
            "error": str(e)
        })

results_df = pd.DataFrame(results)
results_df

Asset directory: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-10-29_803121\803121_2025-10-29_11-19-29
Extracted mouse_id: 803121
Extracted session_datetime: 2025-10-29 11:19:29


Less than 5 trial start or end times found in HARP data. Using SLAP2 frame clock input instead.


Succesfully wrote acquisition JSON to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-10-29_803121\803121_2025-10-29_11-19-29
Asset directory: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-10-30_803121\803121_2025-10-30_11-13-32
Extracted mouse_id: 803121
Extracted session_datetime: 2025-10-30 11:13:32


Less than 5 trial start or end times found in HARP data. Using SLAP2 frame clock input instead.


Succesfully wrote acquisition JSON to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-10-30_803121\803121_2025-10-30_11-13-32
Asset directory: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-10-31_803121\803121_2025-10-31_13-05-26
Extracted mouse_id: 803121
Extracted session_datetime: 2025-10-31 13:05:26


...

Less than 5 trial start or end times found in HARP data. Using SLAP2 frame clock input instead.


Succesfully wrote acquisition JSON to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-11-05_803121\803121_2025-11-05_11-16-57
Asset directory: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-11-06_803121\803121_2025-11-06_12-12-23
Extracted mouse_id: 803121
Extracted session_datetime: 2025-11-06 12:12:23


Less than 5 trial start or end times found in HARP data. Using SLAP2 frame clock input instead.


Succesfully wrote acquisition JSON to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f\803121\2025-11-06_803121\803121_2025-11-06_12-12-23


,session_dir,status,error
0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,success,
1,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,success,
2,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,success,
3,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,success,
4,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,success,
5,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,success,
